In [1]:
%conda install -y git

3 channel Terms of Service accepted
Retrieving notices: done
Channels:
 - defaults
Platform: win-64
Solving environment: done

## Package Plan ##

  environment location: c:\Users\zahra\anaconda3

  added / updated specs:
    - git


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    openssl-3.5.7              |       hbb43b14_0         8.9 MB
    ------------------------------------------------------------
                                           Total:         8.9 MB

The following packages will be UPDATED:

  openssl                                  3.5.6-hbb43b14_0 --> 3.5.7-hbb43b14_0 



openssl-3.5.7        | 8.9 MB    |            |   0% 
openssl-3.5.7        | 8.9 MB    |            |   0% 
openssl-3.5.7        | 8.9 MB    | #####6     |  56% 
openssl-3.5.7        | 8.9 MB    | ########## | 100% 
openssl-3.5.7        | 8.9 MB    | ########## | 100% 
openssl-3.5.7        | 8.9 MB 



==> WARNING: A newer version of conda exists. <==
    current version: 26.1.1
    latest version: 26.5.2

Please update conda by running

    $ conda update -n base -c defaults conda



# >>>>>>>>>>>>>>>>>>>>>> ERROR REPORT <<<<<<<<<<<<<<<<<<<<<<

    Traceback (most recent call last):
      File "C:\Users\zahra\anaconda3\Lib\site-packages\conda\gateways\disk\delete.py", line 144, in unlink_or_rename_to_trash
        os.unlink(path)
        ~~~~~~~~~^^^^^^
    PermissionError: [WinError 5] Access is denied: 'c:\\Users\\zahra\\anaconda3\\Library\\bin\\libcrypto-3-x64.dll.c~'
    
    During handling of the above exception, another exception occurred:
    
    Traceback (most recent call last):
      File "C:\Users\zahra\anaconda3\Lib\site-packages\conda\gateways\disk\delete.py", line 147, in unlink_or_rename_to_trash
        os.rename(path, path + ".conda_trash")
        ~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    FileExistsError: [WinError 183] Cannot create a file when that file alre

In [2]:
%pip install arduino-iot-client requests-oauthlib

Note: you may need to restart the kernel to use updated packages.


In [3]:
from oauthlib.oauth2 import BackendApplicationClient
from requests_oauthlib import OAuth2Session

import iot_api_client as iot
from iot_api_client.rest import ApiException
from iot_api_client.configuration import Configuration
from iot_api_client.api import ThingsV2Api, PropertiesV2Api, SeriesV2Api
from iot_api_client.models import *

import csv
from time import sleep

HOST = "https://api2.arduino.cc"
TOKEN_URL = "https://api2.arduino.cc/iot/v1/clients/token"

# --- CONFIGURATION ---
client_id = "PkgpXiSWBYG3ItmUcBDLtPFKRNHIL2kD" 
client_secret = "ehH5nZK6uz1gYpVWVJDd8slyDS24tguwFmvowCEOGiaOY16CFbbirKwbez9hJYBo" 
org_id = "8a697149-a271-4a3e-9061-6575303ab06b" 

# Update these to the dates you actually want to extract
extract_from = "2026-05-01T00:00:00Z"
extract_to = "2026-05-07T00:00:00Z"
filename = "sensor_data.csv"

# TARGET FILTERING: Add the exact names of your cloud variables here (lowercase recommended for matching)
TARGET_VARIABLES = ["temperature", "humidity", "temp", "hum"]
# ---------------------

def get_token():
    oauth_client = BackendApplicationClient(client_id=client_id)
    oauth = OAuth2Session(client=oauth_client)
    token = oauth.fetch_token(
        token_url=TOKEN_URL,
        client_id=client_id,
        client_secret=client_secret,
        include_client_id=True,
        audience="https://api2.arduino.cc/iot",
        headers={"X-Organization": org_id} if org_id else {}
    )
    return token

def init_client(token):
    client_config = Configuration(HOST)
    client_config.access_token = token.get("access_token")
    if org_id:
        client = iot.ApiClient(client_config, header_name="X-Organization", header_value=org_id)
    else:
        client = iot.ApiClient(client_config)
    return client

def dump_property_data(client, csv_writer, thing_name, prop_name, thing_id, prop_id):
    sleep(1) # Rate limiting courtesy pause
    print(f"Extracting history for: {thing_name} -> {prop_name}")
    series_api = SeriesV2Api(client)
    
    propertyRequest = BatchQueryRawRequestMediaV1(q="property."+prop_id, var_from=extract_from, to=extract_to)
    seriesRequest = BatchQueryRawRequestsMediaV1(resp_version=1, requests=[propertyRequest])
    
    try:
        timeseries = series_api.series_v2_batch_query_raw(seriesRequest)  
        for s in timeseries.responses:
            if not s.times:
                print(f"  No data found for {prop_name} in this date range.")
                continue
            
            # Loop through timestamps and values concurrently
            for i in range(len(s.times)):
                csv_writer.writerow([thing_name, prop_name, s.times[i], s.values[i]])
                
    except ApiException as e:
        print(f"Exception in series extraction for {prop_name}: {e}")

def get_things_and_props(csv_writer):
    token = get_token()
    client = init_client(token)
    things_api = ThingsV2Api(client)
    properties_api = PropertiesV2Api(client)
    todolist = [] 
    
    try:
        things = things_api.things_v2_list()
        for thing in things:
            sleep(1)
            print(f"Checking device: {thing.name}")
            
            properties = properties_api.properties_v2_list(id=thing.id, show_deleted=False)  
            for property in properties:
                name = property.name
                ptype = property.type
                
                # CRITICAL CHANGE: Only grab it if it matches our target sensor names
                if name.lower() in TARGET_VARIABLES:
                    print(f"  -> Target found: {name} ({ptype})")
                    todo = {
                        "thing_id": thing.id,
                        "thing_name": thing.name,
                        "prop_id": property.id,
                        "prop_name": name
                    }
                    todolist.append(todo)
                else:
                    print(f"  (Skipping non-target variable: {name})")

    except ApiException as e:
        print(f"Exception during discovery: {e}")
        return

    # Process the filtered queue
    while len(todolist) != 0:
        todo = todolist.pop()
        dump_property_data(client, csv_writer, todo["thing_name"], todo["prop_name"], todo["thing_id"], todo["prop_id"])

# --- MAIN EXECUTION ---
if __name__ == "__main__":
    with open(filename, 'w', newline='') as outfile:
        writer = csv.writer(outfile)
        # Organized CSV Header
        writer.writerow(["device_name", "sensor_type", "timestamp", "reading_value"])
        
        print("Starting Arduino Cloud Data Extraction...")
        get_things_and_props(writer)
        print(f"Done! Data saved to {filename}")

PermissionError: [Errno 13] Permission denied: 'sensor_data.csv'

In [ ]:
from oauthlib.oauth2 import BackendApplicationClient
from requests_oauthlib import OAuth2Session

import iot_api_client as iot
from iot_api_client.rest import ApiException
from iot_api_client.configuration import Configuration
from iot_api_client.api import ThingsV2Api, PropertiesV2Api, SeriesV2Api
from iot_api_client.models import *

import csv
from time import sleep

HOST = "https://api2.arduino.cc"
TOKEN_URL = "https://api2.arduino.cc/iot/v1/clients/token"

# --- CONFIGURATION ---
client_id = "PkgpXiSWBYG3ItmUcBDLtPFKRNHIL2kD" 
client_secret = "ehH5nZK6uz1gYpVWVJDd8slyDS24tguwFmvowCEOGiaOY16CFbbirKwbez9hJYBo" 
org_id = "8a697149-a271-4a3e-9061-6575303ab06b" 

# --- SPECIFIC TARGET DEVICE ---
TARGET_THING_NAME = "projectA_60307052_60306981"  # <--- Only this device will be processed

# Update these to the dates you actually want to extract
extract_from = "2026-06-02T00:00:00Z"
extract_to = "2026-06-03T00:00:00Z"
filename = "sensor_data.csv"

# TARGET FILTERING: Add the exact names of your cloud variables here (lowercase recommended for matching)
TARGET_VARIABLES = ["temperature", "humidity", "temp", "hum"]
# ---------------------

def get_token():
    oauth_client = BackendApplicationClient(client_id=client_id)
    oauth = OAuth2Session(client=oauth_client)
    token = oauth.fetch_token(
        token_url=TOKEN_URL,
        client_id=client_id,
        client_secret=client_secret,
        include_client_id=True,
        audience="https://api2.arduino.cc/iot",
        headers={"X-Organization": org_id} if org_id else {}
    )
    return token

def init_client(token):
    client_config = Configuration(HOST)
    client_config.access_token = token.get("access_token")
    if org_id:
        client = iot.ApiClient(client_config, header_name="X-Organization", header_value=org_id)
    else:
        client = iot.ApiClient(client_config)
    return client

def dump_property_data(client, csv_writer, thing_name, prop_name, thing_id, prop_id):
    sleep(1) # Rate limiting courtesy pause
    print(f"Extracting history for: {thing_name} -> {prop_name}")
    series_api = SeriesV2Api(client)
    
    propertyRequest = BatchQueryRawRequestMediaV1(q="property."+prop_id, var_from=extract_from, to=extract_to)
    seriesRequest = BatchQueryRawRequestsMediaV1(resp_version=1, requests=[propertyRequest])
    
    try:
        timeseries = series_api.series_v2_batch_query_raw(seriesRequest)  
        for s in timeseries.responses:
            if not s.times:
                print(f"  No data found for {prop_name} in this date range.")
                continue
            
            # Loop through timestamps and values concurrently
            for i in range(len(s.times)):
                csv_writer.writerow([thing_name, prop_name, s.times[i], s.values[i]])
                
    except ApiException as e:
        print(f"Exception in series extraction for {prop_name}: {e}")

def get_things_and_props(csv_writer):
    token = get_token()
    client = init_client(token)
    things_api = ThingsV2Api(client)
    properties_api = PropertiesV2Api(client)
    todolist = [] 
    
    try:
        things = things_api.things_v2_list()
        for thing in things:
            # ─── MODIFICATION: Filter out all other Things ───
            if thing.name != TARGET_THING_NAME:
                continue # Skip directly to the next thing in the list
                
            sleep(1)
            print(f"Checking device: {thing.name}")
            
            properties = properties_api.properties_v2_list(id=thing.id, show_deleted=False)  
            for property in properties:
                name = property.name
                ptype = property.type
                
                # Only grab it if it matches our target sensor names
                if name.lower() in TARGET_VARIABLES:
                    print(f"  -> Target found: {name} ({ptype})")
                    todo = {
                        "thing_id": thing.id,
                        "thing_name": thing.name,
                        "prop_id": property.id,
                        "prop_name": name
                    }
                    todolist.append(todo)
                else:
                    print(f"  (Skipping non-target variable: {name})")

    except ApiException as e:
        print(f"Exception during discovery: {e}")
        return

    # Process the filtered queue
    while len(todolist) != 0:
        todo = todolist.pop()
        dump_property_data(client, csv_writer, todo["thing_name"], todo["prop_name"], todo["thing_id"], todo["prop_id"])

# --- MAIN EXECUTION ---
if __name__ == "__main__":
    with open(filename, 'w', newline='') as outfile:
        writer = csv.writer(outfile)
        # Organized CSV Header
        writer.writerow(["device_name", "sensor_type", "timestamp", "reading_value"])
        
        print("Starting Arduino Cloud Data Extraction...")
        get_things_and_props(writer)
        print(f"Done! Data saved to {filename}")

Starting Arduino Cloud Data Extraction...
Checking device: projectA_60307052_60306981
  -> Target found: humidity (FLOAT)
  -> Target found: temperature (FLOAT)
Extracting history for: projectA_60307052_60306981 -> temperature
Extracting history for: projectA_60307052_60306981 -> humidity
Done! Data saved to sensor_data.csv
